# 10 -- Ambient and amplification

Step-by-step ambient analysis, HVSR and amplification (windowed).

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 32.25 GB / avail 1.26 GB (96%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
config = {"Fs": ds.fs, "STA": 1.0, "LTA": 30.0,
          "vent": 20.0, "vmin": 0.0, "vmax": 1000.0,   # wide -> keep all windows
          "p": 0.05, "bexp": 40, "f1": 0.1, "f2": 25.0, "vent_seismic": False}
config

{'Fs': 252.58854665265045,
 'STA': 1.0,
 'LTA': 30.0,
 'vent': 20.0,
 'vmin': 0.0,
 'vmax': 1000.0,
 'p': 0.05,
 'bexp': 40,
 'f1': 0.1,
 'f2': 25.0,
 'vent_seismic': False}

## Ambient, step by step

In [4]:
sig = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9).signal()
amb = sig.ambient(config, component="x")
amb.sta_lta(); amb.select_windows(); amb.taper(); amb.fft(); amb.smooth(); amb.average()
print("windows kept:", amb.windows_signal.shape[1] if amb.windows_signal.ndim > 1 else 1)
print("dominant period:", round(float(amb.dominant_period),3), "s")

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[signal] MOF00135 n=149240 dt=0.004021 comps=all


- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.2498 s)
windows kept: 28
dominant period: 1.25 s


## Compact form (call average())

In [5]:
amb2 = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9).signal().ambient(config, "x")
amb2.average()
print("mean spectrum points:", amb2.mean_spectrum.shape[0])

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[signal] MOF00135 n=149240 dt=0.004021 comps=all
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed
- smooth: Konno-Ohmachi smoothing applied (bexp=40)


- average: mean spectrum computed (T=1.2498 s)
mean spectrum points: 2525


## HVSR (Nakamura)

In [6]:
from asdea_sensors.ambient import hvsr
s = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9).signal(components="all")
hv = hvsr.compute(s.acc_x, s.acc_y, s.acc_z, config, combine="geometric")
print("f0 =", round(float(hv["f0"]), 2), "Hz")

[signal] MOF00135 n=149240 dt=0.004021 comps=all
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.2498 s)
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.5382 s)
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.1109 s)
f0 = 93.51 Hz


## Amplification (Fourier basis)

In [7]:
from asdea_sensors.ambient import amplification
ref = ds.MOF00134.window(WSTART, WLEN).baseline().filter(0.1, 24.9).signal(components="x")
others = {d: ds.device(d).window(WSTART, WLEN).baseline().filter(0.1, 24.9).signal(components="x").acc_x
          for d in ["MNAT0031", "MOF00135"]}
amp = amplification.compute(ref.acc_x, others, ref.dt, basis="fourier", config=config)
print("amplification basis:", amp["basis"], " devices:", list(amp["ratios"].keys()))

[signal] MOF00134 n=149340 dt=0.004018 comps=x


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0031' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[signal] MNAT0031 n=151560 dt=0.003959 comps=x


[signal] MOF00135 n=149240 dt=0.004021 comps=x
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=3.9994 s)
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 29 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=3.9994 s)
- sta_lta: ratio computed (STA=1 s, LTA=30 s)
- select_windows: 28 window(s) selected
- taper: Tukey taper applied (p=0.05)
- fft: per-window FFT computed


- smooth: Konno-Ohmachi smoothing applied (bexp=40)
- average: mean spectrum computed (T=1.2498 s)
amplification basis: fourier  devices: ['MNAT0031', 'MOF00135']
